# Scratch README Atlas Demo

Self-contained synthetic 2D demo for generating a README figure. This notebook is intentionally temporary and should not be committed unless we decide to keep it.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
from sklearn.datasets import make_moons

ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from certcf import CertCFAtlas
from certcf.eps_strategies import NearestOppositeClassClearanceStrategy
from certcf.geometry import make_polygon

SEED = 7
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
OUTPUT_DIR = ROOT / 'assets'
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
def make_nonlinear_moons(n_samples: int = 420, noise: float = 0.075):
    X, y = make_moons(n_samples=n_samples, noise=noise, random_state=SEED)
    X = X.astype(np.float32)
    y = y.astype(np.int64)

    # Gentle affine transform: wider canvas and a slight tilt for a nicer README composition.
    X[:, 0] = 1.45 * X[:, 0] - 0.45
    X[:, 1] = 1.25 * X[:, 1]
    theta = np.deg2rad(-8.0)
    rot = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]], dtype=np.float32)
    X = X @ rot.T
    order = rng.permutation(len(X))
    return X[order], y[order]

X, y = make_nonlinear_moons()
print(X.shape, np.bincount(y))


In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 24),
            nn.ReLU(),
            nn.Linear(24, 24),
            nn.ReLU(),
            nn.Linear(24, 2),
        )

    def forward(self, x):
        return self.net(x)

model = TinyClassifier()
optimizer = torch.optim.AdamW(model.parameters(), lr=2.5e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()
X_t = torch.from_numpy(X)
y_t = torch.from_numpy(y)

for epoch in range(900):
    optimizer.zero_grad(set_to_none=True)
    logits = model(X_t)
    loss = loss_fn(logits, y_t)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred = model(X_t).argmax(dim=1)
    acc = (pred == y_t).float().mean().item()
print(f'train accuracy: {acc:.3f}')


In [ ]:
# Use a small, balanced support set so the figure is readable and LiRPA stays fast.
support_idx = []
for cls in [0, 1]:
    cls_idx = np.flatnonzero(y == cls)
    support_idx.extend(rng.choice(cls_idx, size=42, replace=False).tolist())
support_idx = np.array(support_idx)
X_support = X[support_idx]
y_support = y[support_idx]

dataset = TensorDataset(torch.from_numpy(X_support), torch.from_numpy(y_support))

atlas = CertCFAtlas(
    model=model,
    dataset=dataset,
    device='cpu',
    norm=1,
    distance_norm=1,
    eps_strategy=NearestOppositeClassClearanceStrategy(alpha=0.40),
    batch_size=64,
    classification_margin=1.0e-4,
    adaptive_eps=True,
    adaptive_eps_shrink_factor=0.5,
    adaptive_eps_max_shrinks=4,
)
atlas.build(build_unions=True, verbose=True)


In [ ]:
# Pick a source point and compute a certified counterfactual toward the other class.
query_candidates = np.flatnonzero((y_support == 0) & (X_support[:, 0] > -0.65) & (X_support[:, 0] < 0.25))
if len(query_candidates) == 0:
    query_candidates = np.flatnonzero(y_support == 0)
query_idx = query_candidates[np.argmin(np.abs(X_support[query_candidates, 1] - 0.75))]
x_query = X_support[query_idx].astype(np.float32)
target_class = 1

cf_result = atlas.find_counterfactual(
    x_query,
    target_class=target_class,
    method='nearest_anchor',
    query_k_candidates=8,
)
if not cf_result.success:
    cf_result = atlas.find_counterfactual(x_query, target_class=target_class, method='sorted')

print({'success': cf_result.success, 'distance': cf_result.distance, 'anchor_idx': cf_result.anchor_idx})
x_cf = np.asarray(cf_result.x_cf, dtype=np.float32)


In [ ]:
def plot_readme_atlas(atlas, X, y, X_support, y_support, x_query=None, x_cf=None):
    colors = {0: '#266DD3', 1: '#E94F37'}
    fills = {0: '#6EA8FE', 1: '#F28E7C'}

    fig, ax = plt.subplots(figsize=(8.2, 5.2), dpi=180)

    # Soft decision background.
    pad = 0.75
    x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 350), np.linspace(y_min, y_max, 260))
    grid = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
    with torch.no_grad():
        prob = torch.softmax(model(torch.from_numpy(grid)), dim=1)[:, 1].numpy().reshape(xx.shape)
    ax.contourf(xx, yy, prob, levels=np.linspace(0, 1, 16), cmap='RdBu_r', alpha=0.16)
    ax.contour(xx, yy, prob, levels=[0.5], colors='#20242A', linewidths=1.2, alpha=0.65)

    # Certified atlas union per class.
    for cls in atlas.class_labels:
        union = atlas._class_unions[int(cls)]
        geoms = [union] if union.geom_type == 'Polygon' else list(union.geoms)
        for geom in geoms:
            if geom.is_empty or geom.area <= 1e-10:
                continue
            xs, ys = geom.exterior.xy
            ax.fill(xs, ys, color=fills[int(cls)], alpha=0.34, zorder=2)
            ax.plot(xs, ys, color=colors[int(cls)], lw=1.3, alpha=0.88, zorder=3)

    # Full data and atlas anchors.
    for cls in [0, 1]:
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], s=13, color=colors[cls], alpha=0.18, edgecolor='none', zorder=1)
        smask = y_support == cls
        ax.scatter(X_support[smask, 0], X_support[smask, 1], s=28, color=colors[cls], alpha=0.92,
                   edgecolor='white', linewidth=0.45, zorder=4, label=f'class {cls} anchors')

    if x_query is not None and x_cf is not None:
        x_query = np.asarray(x_query, dtype=float)
        x_cf = np.asarray(x_cf, dtype=float)
        ax.annotate(
            '',
            xy=x_cf,
            xytext=x_query,
            arrowprops=dict(
                arrowstyle='-|>',
                lw=2.6,
                color='#20242A',
                shrinkA=6,
                shrinkB=6,
                mutation_scale=18,
            ),
            zorder=7,
        )
        ax.scatter([x_query[0]], [x_query[1]], s=88, marker='o', color='#111827',
                   edgecolor='white', linewidth=1.0, zorder=8, label='query')
        ax.scatter([x_cf[0]], [x_cf[1]], s=105, marker='*', color='#FFD166',
                   edgecolor='#111827', linewidth=0.8, zorder=9, label='CertCF')

    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax.legend(loc='upper left', frameon=True, framealpha=0.92, fontsize=9)
    for spine in ax.spines.values():
        spine.set_color('#343A40')
        spine.set_linewidth(0.8)
    fig.tight_layout()
    return fig, ax

fig, ax = plot_readme_atlas(atlas, X, y, X_support, y_support, x_query=x_query, x_cf=x_cf)
out = OUTPUT_DIR / 'readme_certcf_atlas_demo.png'
fig.savefig(out, bbox_inches='tight')
print(out)
plt.show()
